# ARIMA model (the [`pmdarima`](https://alkaline-ml.com/pmdarima/index.html#) package)

ARIMA(p,d,q) specification (ARIMA = AutoRegressive Integrated Moving Average)

$$
\begin{aligned}
	\Delta ^d y_t &= (\alpha_0+\alpha_1 t)+\sum_{j=1}^p\phi_j\Delta^d y_{t-j}+u_t+\sum_{s=1}^q\theta_s u_{t-s} &
	u_t&\sim WN(0,\sigma^2)
\end{aligned}
$$
where
* p - order of the autoregressive part
* d - order of integration
* q - order of the moving average part 
* $\alpha_0$ - constant (d=0) or drift (d>0)

Specification using the lag operator

$$
\begin{aligned}
	\phi_p(L)(1-L)^dy_t&=(\alpha_0+\alpha_1t)+\theta_q(L)u_t & u_t&\sim WN(0,\sigma^2)
\end{aligned}
$$
where the polynomials
$$
\begin{aligned}
	\phi_p(z)&=1-\phi_1z-\cdots-\phi_pz^p & \theta_q(z)&=1+\theta_1z+\cdots+\theta_qz^q
\end{aligned}
$$

In [ ]:
import numpy as np
import pandas as pd

import pmdarima as pm

import pandas_datareader.data as web

# visualization settings
import matplotlib.pyplot as plt

# Do not show Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

To fit, we use the [ARIMA](https://alkaline-ml.com/pmdarima/modules/generated/pmdarima.arima.ARIMA.html#pmdarima.arima.ARIMA) class

When specifying the model, you need to set the `order` (model order) and `trend` parameters

|Parameters|`trend`|
|-|-|
|$\alpha_0=\alpha_1=0$|'n'|
|$\alpha_0\ne0,\alpha_1=0$|'c'|
|$\alpha_0,\alpha_1\ne0$|'ct'|


## Fitting an ARIMA of a given order

Load weekly data from the [`FRED`](https://fred.stlouisfed.org/) database on the 3-Month Treasury Bill Secondary Market Rate (Symbol [`WTB3MS`](https://fred.stlouisfed.org/series/WTB3MS)) from 2000-01-01 to 2023-12-31 and create a dataframe `rate`

In [ ]:
y = web.DataReader(name='WTB3MS', data_source='fred', start='2000-01-01', end='2023-12-31')

Fit an ARIMA(2,1,3) model without drift

Specification

$$
	(1-\phi_1L-\phi_2 L^2)(1-L) y_t=u_t+\theta_1 u_{t-1}+\theta_2 u_{t-2}+\theta_3 u_{t-3}
$$

In [ ]:
arima = pm.ARIMA(order=(2,1,3), trend='n')
arima.fit(y)
arima.summary()

In [ ]:
arima.plot_diagnostics()
plt.show()

## Forecasting

Build a forecast using an ARIMA(1,1,1) model with drift

Specification

$$
	(1-\phi L)(1-L) y_t=\alpha_0+u_t+\theta u_{t-1}
$$

Numerical forecast for 10 periods

In [ ]:
arima = pm.ARIMA(order=(1,1,1), trend='c')
# fit the model and forecast
forecasts = arima.fit_predict(y, n_periods=10)
forecasts

Numerical forecast and confidence intervals

In [ ]:
forecasts, conf_int = arima.predict(n_periods=10, return_conf_int=True, alpha=0.05)
forecasts, conf_int

## Automatic selection of the model order

use the [`auto_arima`](https://alkaline-ml.com/pmdarima/modules/generated/pmdarima.arima.auto_arima.html#pmdarima.arima.auto_arima) method

Basic parameters of automatic selection

|Criterion|`auto_arima`|Values|
|-|-|-|
|Information criterion|`information_criterion`|aic (default), aicc, bic, hqic |
|unit root test|`test`|kpss (default), adf, pp|
|max order d|`max_d`| 2 by default |
|max order p,q|` max_p`, ` max_q`|5 by default|
|seasonal model|`seasonal`|True by default|
|significance level|`alpha`|0.05 by default|

__Note__: whether to include drift/constant/trend in the model is selected automatically

In [ ]:
arima_opt = pm.auto_arima(y, information_criterion='bic', seasonal=False)
arima_opt.get_params()

Forecast using the optimal model

In [ ]:
arima_opt.predict(n_periods=10)